# 4. 이미지 업로드해서 직접 확인해보기

원하는 영수증 사진을 업로드하면, **베이스 모델(파인튜닝 전) vs 파인튜닝한 모델**이 어떻게 다르게
읽는지 바로 비교해볼 수 있습니다. 검출(EasyOCR)은 공통으로 쓰고, 인식만 두 모델로 각각 돌립니다.


## 0. 환경 설정 (모델 두 개 다 로드)

In [ ]:
import os

os.environ.setdefault("USE_TF", "0")

import torch
import easyocr
import numpy as np
from PIL import Image
from IPython.display import display
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

PROJECT_DIR = os.getcwd()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {
    "베이스(파인튜닝 전)": os.path.join(PROJECT_DIR, "model", "trocr_base"),
    "파인튜닝 후": os.path.join(PROJECT_DIR, "model", "trocr_v3"),
}

loaded = {}
for name, path in MODELS.items():
    processor = TrOCRProcessor.from_pretrained(path)
    model = VisionEncoderDecoderModel.from_pretrained(path).to(DEVICE)
    model.eval()
    loaded[name] = (model, processor)
    print(f"로드 완료: {name}")

reader = easyocr.Reader(["ko", "en"], gpu=torch.cuda.is_available())
print("EasyOCR 검출기 로드 완료")


## 1. 검출+인식 파이프라인 함수

In [ ]:
import re

# 검출 박스가 이 픽셀 크기보다 작으면 노이즈(점, 얼룩 등)일 가능성이 높아 인식을 시도하지 않고 건너뜀
MIN_CROP_WIDTH = 6
MIN_CROP_HEIGHT = 8
# 평균 토큰 확률이 이보다 낮으면 화면에 ⚠️로 표시(오답 확정은 아니고 검수가 필요하다는 신호)
CONF_THRESHOLD = 0.5


def box_rect(box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]
    return min(xs), min(ys), max(xs), max(ys)


def sort_reading_order(boxes):
    rects = [box_rect(b) for b in boxes]
    order = sorted(range(len(boxes)), key=lambda i: (rects[i][1] + rects[i][3]) / 2)
    heights = [r[3] - r[1] for r in rects]
    avg_h = sum(heights) / len(heights)
    lines, current, current_y = [], [order[0]], (rects[order[0]][1] + rects[order[0]][3]) / 2
    for idx in order[1:]:
        yc = (rects[idx][1] + rects[idx][3]) / 2
        if abs(yc - current_y) < avg_h * 0.6:
            current.append(idx)
        else:
            lines.append(current)
            current = [idx]
            current_y = yc
    lines.append(current)
    result = []
    for line in lines:
        result.extend(sorted(line, key=lambda i: rects[i][0]))
    return [(boxes[i], rects[i]) for i in result]


def normalize_number_spacing(text):
    """'37, 000' -> '37,000' 처럼 천 단위 쉼표 뒤에 낀 어색한 공백만 정리."""
    return re.sub(r"(\d),\s+(?=\d)", r"\1,", text)


def recognize_with(model, processor, crop, max_length=64):
    """인식 결과 텍스트와 함께, 생성된 토큰들의 평균 확률(0~1)을 신뢰도로 반환."""
    pixel_values = processor(images=crop, return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            max_length=max_length,
            output_scores=True,
            return_dict_in_generate=True,
        )
    text = processor.batch_decode(outputs.sequences, skip_special_tokens=True)[0]
    text = normalize_number_spacing(text)

    # beam search로 생성된 경우 beam_indices가 있어야 점수가 정확히 계산됨(greedy면 None이라 자동 무시됨)
    transition_scores = model.compute_transition_scores(
        outputs.sequences,
        outputs.scores,
        beam_indices=getattr(outputs, "beam_indices", None),
        normalize_logits=True,
    )
    valid_scores = transition_scores[torch.isfinite(transition_scores)]
    confidence = valid_scores.exp().mean().item() if len(valid_scores) else 0.0
    return text, confidence


def format_result(text, confidence):
    if not text.strip():
        return "(인식 실패)"
    return text if confidence >= CONF_THRESHOLD else f"{text} ⚠️"


def compare_on_image(pil_image):
    """영수증 사진 한 장 -> 베이스/파인튜닝 모델 결과를 줄별로 나란히 출력.
    ⚠️ 표시는 모델이 스스로 확신하지 못한(평균 토큰 확률이 낮은) 줄이라는 뜻입니다."""
    display(pil_image)
    boxes_raw = reader.detect(np.array(pil_image))[0][0]
    boxes = [[[x0, y0], [x1, y0], [x1, y1], [x0, y1]] for x0, x1, y0, y1 in boxes_raw]
    ordered = sort_reading_order(boxes)

    print(f"검출된 줄: {len(ordered)}개\n")
    for _, rect in ordered:
        x0, y0, x1, y1 = [int(v) for v in rect]
        crop = pil_image.crop((max(0, x0), max(0, y0), x1, y1))
        if crop.width < MIN_CROP_WIDTH or crop.height < MIN_CROP_HEIGHT:
            continue
        row = []
        for name, (model, processor) in loaded.items():
            text, confidence = recognize_with(model, processor, crop)
            row.append(f"{name}: {format_result(text, confidence)}")
        print("  |  ".join(row))

## 2. 이미지 업로드

아래 셀을 실행하면 업로드 버튼이 나옵니다. 영수증 사진을 올려보세요(파일 선택 후 자동으로
분석이 시작됩니다). 인터넷 없이도 되고, 이미 준비된 샘플(`samples/image.png`)로 먼저 테스트해봐도
됩니다(3번 셀).


In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
import io

uploader = widgets.FileUpload(accept="image/*", multiple=False)
output = widgets.Output()


def on_upload_change(change):
    with output:
        clear_output()
        if not uploader.value:
            return
        # ipywidgets 버전에 따라 dict 또는 tuple 형태라 둘 다 처리
        item = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]
        content = item["content"]
        pil_image = Image.open(io.BytesIO(content)).convert("RGB")
        compare_on_image(pil_image)


uploader.observe(on_upload_change, names="value")
display(uploader, output)


## 3. (참고) 준비된 샘플 이미지로 먼저 테스트해보기

In [ ]:
sample_path = os.path.join(PROJECT_DIR, "samples", "image.png")
compare_on_image(Image.open(sample_path).convert("RGB"))
